In [ ]:
#!/usr/bin/env python3
"""
================================================================================
WIG to Gene Expression Matrix Converter For GSE161360_RAW.tar, only for FN (F. nucleatum nucleatum ATCC 25586)
================================================================================
Check read me!!!
------
1. Download GSE161360_RAW.tar from GEO
2. Extract the tar file, you do know that only GSM4905285 to GSM4905302 will be processed by this script. The rest will be ignored.
3. Run this script in the directory containing the WIG files

OUTPUT:
-------
- pai2_expression_matrix.csv (raw coverage values)
- pai2_expression_log2.csv (log2-transformed, ready for bnlearn)
- pai2_gene_coordinates.csv (gene coordinates used)

If you want to do other PAI, Just change PAI2-->PAI1 for exxample
================================================================================
"""

import os
import gzip
import numpy as np
import pandas as pd
from glob import glob

# ============================================================================
# STEP 1: DEFINE PAI2 GENE COORDINATES
# ============================================================================
# These coordinates are from NCBI GenBank AE009951.2

# --- PAI1: FN1878–FN1887 (10 genes, 366,847–372,320 bp) ---
# PAI1_GENES = {
#     'FN1878': {'start': 366847,  'end': 368001,  'strand':  1},
#     'FN1879': {'start': 368014,  'end': 368310,  'strand':  1},
#     'FN1880': {'start': 368323,  'end': 368895,  'strand':  1},
#     'FN1881': {'start': 368908,  'end': 369537,  'strand': -1},  # minus strand
#     'FN1882': {'start': 369550,  'end': 369906,  'strand':  1},
#     'FN1883': {'start': 369919,  'end': 370827,  'strand':  1},
#     'FN1884': {'start': 370840,  'end': 371124,  'strand': -1},  # minus strand
#     'FN1885': {'start': 371137,  'end': 371916,  'strand':  1},  # Hemolysin III
#     'FN1886': {'start': 371929,  'end': 372213,  'strand':  1},  # lipoprotein
#     'FN1887': {'start': 372226,  'end': 372320,  'strand':  1},  # transposase
# }

PAI2_GENES = {
    'FN0834': {'start': 1496612, 'end': 1498148, 'strand': 1},
    'FN0835': {'start': 1498328, 'end': 1498925, 'strand': 1},
    'FN0836': {'start': 1498941, 'end': 1499238, 'strand': 1},
    'FN0837': {'start': 1499264, 'end': 1500251, 'strand': 1},
    'FN0838': {'start': 1500886, 'end': 1501288, 'strand': 1},
    'FN0839': {'start': 1501333, 'end': 1501516, 'strand': -1},
    'FN0840': {'start': 1501500, 'end': 1502010, 'strand': 1},
    'FN0841': {'start': 1502240, 'end': 1502864, 'strand': 1},
    'FN0842': {'start': 1502911, 'end': 1503145, 'strand': -1},
    'FN0843': {'start': 1503275, 'end': 1503476, 'strand': -1},
    'FN0844': {'start': 1503669, 'end': 1503834, 'strand': 1},
    'FN0845': {'start': 1504074, 'end': 1504284, 'strand': 1},
    'FN0846': {'start': 1504381, 'end': 1505296, 'strand': 1},
    'FN0847': {'start': 1505392, 'end': 1507192, 'strand': 1},
    'FN0848': {'start': 1507208, 'end': 1507823, 'strand': 1},
    'FN0849': {'start': 1507906, 'end': 1509052, 'strand': 1},
    'FN0850': {'start': 1509052, 'end': 1509643, 'strand': 1},
    'FN0851': {'start': 1509659, 'end': 1510328, 'strand': 1},
    'FN0852': {'start': 1510363, 'end': 1510963, 'strand': 1},
    'FN0853': {'start': 1511011, 'end': 1512397, 'strand': -1},
    'FN0854': {'start': 1512413, 'end': 1513577, 'strand': -1},
    'FN0855': {'start': 1513597, 'end': 1514752, 'strand': -1},
    'FN0856': {'start': 1514778, 'end': 1516614, 'strand': -1},
    'FN0857': {'start': 1516642, 'end': 1519012, 'strand': -1},
    'FN0858': {'start': 1519076, 'end': 1520597, 'strand': -1},
    'FN0859': {'start': 1520809, 'end': 1521319, 'strand': -1},
    'FN0860': {'start': 1521395, 'end': 1521653, 'strand': -1},
    'FN0861': {'start': 1521827, 'end': 1522049, 'strand': -1},
    'FN0862': {'start': 1522215, 'end': 1522461, 'strand': -1},
    'FN0863': {'start': 1522487, 'end': 1522664, 'strand': -1},
    'FN0864': {'start': 1522651, 'end': 1522990, 'strand': -1},
    'FN0865': {'start': 1523129, 'end': 1523855, 'strand': -1},
}

# --- PAI3: FN1313–FN1345 (33 genes, 1,967,193–1,999,047 bp) ---
# PAI3_GENES = {
#     'FN1313': {'start': 1967193, 'end': 1968617,  'strand':  1},  # OppA
#     'FN1314': {'start': 1968630, 'end': 1969049,  'strand': -1},
#     'FN1315': {'start': 1969062, 'end': 1969556,  'strand': -1},
#     'FN1316': {'start': 1969569, 'end': 1970009,  'strand': -1},
#     'FN1317': {'start': 1970022, 'end': 1970618,  'strand': -1},  # sigma factor
#     'FN1318': {'start': 1970631, 'end': 1972613,  'strand': -1},  # SigA/RpoD
#     'FN1319': {'start': 1972626, 'end': 1973219,  'strand': -1},
#     'FN1320': {'start': 1973232, 'end': 1973651,  'strand': -1},
#     'FN1321': {'start': 1973664, 'end': 1974716,  'strand': -1},  # PilR
#     'FN1322': {'start': 1974729, 'end': 1975793,  'strand': -1},  # MucP
#     'FN1323': {'start': 1975806, 'end': 1976468,  'strand': -1},  # thymidylate kinase
#     'FN1324': {'start': 1976481, 'end': 1977617,  'strand': -1},  # Dxr
#     'FN1325': {'start': 1977630, 'end': 1978268,  'strand': -1},  # CpsB
#     'FN1326': {'start': 1978281, 'end': 1979129,  'strand': -1},  # CpsA
#     'FN1327': {'start': 1979142, 'end': 1980134,  'strand': -1},  # dimethylallyltransferase
#     'FN1328': {'start': 1980147, 'end': 1980614,  'strand': -1},
#     'FN1329': {'start': 1980627, 'end': 1981484,  'strand': -1},  # RsmD methyltransferase
#     'FN1330': {'start': 1981497, 'end': 1982594,  'strand': -1},  # QueA
#     'FN1331': {'start': 1982607, 'end': 1983218,  'strand': -1},  # PrmC
#     'FN1332': {'start': 1983231, 'end': 1984127,  'strand': -1},  # RF-1
#     'FN1333': {'start': 1984140, 'end': 1984607,  'strand': -1},
#     'FN1334': {'start': 1984620, 'end': 1985396,  'strand': -1},  # N-acetylmuramoyl amidase
#     'FN1335': {'start': 1985409, 'end': 1985720,  'strand': -1},  # YajC
#     'FN1336': {'start': 1985733, 'end': 1986218,  'strand': -1},  # FadA cluster
#     'FN1337': {'start': 1986231, 'end': 1986614,  'strand': -1},  # FadA cluster
#     'FN1338': {'start': 1986627, 'end': 1987268,  'strand': -1},  # FadA cluster
#     'FN1339': {'start': 1991809, 'end': 1993290,  'strand':  1},  # transposase (+ strand)
#     'FN1340': {'start': 1993303, 'end': 1994631,  'strand': -1},  # GltX
#     'FN1341': {'start': 1994644, 'end': 1995537,  'strand': -1},  # RF-2
#     'FN1342': {'start': 1995550, 'end': 1995966,  'strand': -1},  # AcpS
#     'FN1343': {'start': 1995979, 'end': 1996614,  'strand': -1},
#     'FN1344': {'start': 1997660, 'end': 1998094,  'strand':  1},  # (+ strand)
#     'FN1345': {'start': 1998107, 'end': 1999047,  'strand': -1},
# }


# ===============================
# STEP 2: FUNCTIONS TO PARSE WIG FILES
# ===============================

def parse_wig_file(filepath):
    coverage = {}
    
    # Handle both gzipped and plain files
    if filepath.endswith('.gz'):
        f = gzip.open(filepath, 'rt')
    else:
        f = open(filepath, 'r')
    
    try:
        for line in f:
            line = line.strip()
            # Skip header lines
            if line.startswith('track') or line.startswith('variableStep'):
                continue
            if line:
                parts = line.split()
                if len(parts) == 2:
                    pos = int(parts[0])
                    val = float(parts[1])
                    coverage[pos] = val
    finally:
        f.close()
    
    return coverage


def get_gene_expression(coverage_fwd, coverage_rev, start, end, strand):

    # Select appropriate strand
    coverage = coverage_fwd if strand == 1 else coverage_rev
    
    # Sum coverage across gene
    total = 0
    for pos in range(start, end + 1):
        total += coverage.get(pos, 0)
    
    # Return average coverage (normalized by gene length)
    gene_length = end - start + 1
    return total / gene_length


# ============================================================================
# STEP 3: MAIN PROCESSING
# ============================================================================

def main():
    print("=" * 70)
    print("WIG to Gene Expression Matrix Converter")
    print("=" * 70)
    
    # Find WIG files for FNN (F. nucleatum nucleatum ATCC 25586)
    # Pattern: GSM*_FNN_*_forward.wig.gz
    fwd_files = sorted(glob("*_FNN_*_forward.wig.gz"))
    
    if not fwd_files:
        # Try without .gz extension
        fwd_files = sorted(glob("*_FNN_*_forward.wig"))
    
    if not fwd_files:
        print("ERROR: Check if your WIG files are in the same file with your code.")
        print("Can't find.")
        print("Expected pattern: GSM*_FNN_*_forward.wig.gz")
        return
    
    print(f"\nFound {len(fwd_files)} FN samples")
    
    # Process each sample
    expression_data = {}
    
    for fwd_file in fwd_files:
        # Get corresponding reverse file
        rev_file = fwd_file.replace('_forward.', '_reverse.')
        
        if not os.path.exists(rev_file):
            print(f"Missing reverse file for {fwd_file}")
            continue
        
        # Extract sample name
        basename = os.path.basename(fwd_file)
        parts = basename.replace('.wig.gz', '').replace('.wig', '').split('_')
        sample_id = '_'.join(parts[1:-1])  # Remove GSM and forward/reverse
        
        print(f"Processing: {sample_id}...")
        
        # Parse WIG files
        cov_fwd = parse_wig_file(fwd_file)
        cov_rev = parse_wig_file(rev_file)
        
        # Extract expression for each PAI2 gene
        sample_expr = {}
        for gene, coords in PAI2_GENES.items():
            expr = get_gene_expression(
                cov_fwd, cov_rev,
                coords['start'], coords['end'], coords['strand']
            )
            sample_expr[gene] = expr
        
        expression_data[sample_id] = sample_expr
    
    # Create expression matrix
    expr_df = pd.DataFrame(expression_data).T
    expr_df.index.name = 'sample'
    
    # Sort columns by gene number
    expr_df = expr_df[sorted(expr_df.columns, key=lambda x: int(x[2:]))]
    
    print(f"\nExpression matrix: {expr_df.shape[0]} samples × {expr_df.shape[1]} genes")
    
    # Save raw expression
    expr_df.to_csv("pai2_expression_matrix.csv")
    print("Ready to see: pai2_expression_matrix.csv")
    
    # Log2 transform (with pseudocount)
    # Take absolute values first (strand-specific coverage can be negative)
    expr_log = np.log2(expr_df.abs() + 1)
    expr_log.to_csv("pai2_expression_log2.csv")
    print("Ready to see: pai2_expression_log2.csv (log2-transformed, ready for bnlearn)")
    
    # Save gene coordinates
    coords_df = pd.DataFrame(PAI2_GENES).T
    coords_df.index.name = 'locus_tag'
    coords_df.to_csv("pai2_gene_coordinates.csv")
    print("Ready to see: pai2_gene_coordinates.csv")
    
    print("\n" + "=" * 70)
    print("Ok. This is it. Use pai2_expression_log2.csv as input for bnlearn")
    print("=" * 70)


if __name__ == "__main__":
    main()